# Sales dataset
Load sales for 2025 and 2026. Keep this notebook and `sales_dataset.csv` in the same folder.

In [1]:
import pandas as pd

sales = pd.read_csv("sales_dataset.csv")
sales

,Year,Customer,product,Quantity,Price
0,2025,Customer A,Notebook,100,5.0
1,2025,Customer A,Pen,200,2.0
2,2025,Customer A,Folder,80,3.0
3,2025,Customer B,Notebook,150,5.0
4,2025,Customer B,Pen,300,2.0
5,2025,Customer B,Folder,100,3.0
6,2025,Customer C,Notebook,80,5.0
7,2025,Customer C,Pen,150,2.0
8,2025,Customer C,Folder,60,3.0
9,2025,Customer D,Notebook,120,5.0


## Classical price and volume/mix analysis

Following Sections 4.1–4.4 of `Price-Volume-Overview.ipynb`, 2025 is the previous year and 2026 is the current year. For this synthetic example, all customer–product relationships are assumed to be eligible base business, and fixed charges are zero.

- Sales = quantity × unit price.
- Average price = total sales / total quantity for each customer, product, and year (a quantity-weighted price if multiple transactions exist).
- Price effect = (average price 2026 − average price 2025) × quantity 2025.
- Volume mix effect = average price 2026 × (quantity 2026 − quantity 2025).

The price–quantity interaction is included in volume/mix, as in the overview. Each row must reconcile: **Sales 2026 − Sales 2025 = Price effect + Volume mix effect**.

In [2]:
import pandas as pd

sales = pd.read_csv("sales_dataset.csv")

# Aggregate transactions before calculating weighted average prices.
annual_sales = (
    sales.rename(columns={"product": "Product"})
    .assign(Sales=sales["Quantity"] * sales["Price"])
    .groupby(["Customer", "Product", "Year"], as_index=False)
    .agg(Sales=("Sales", "sum"), Qty=("Quantity", "sum"))
)

# This example requires comparable business and positive quantities in both years.
previous = annual_sales.loc[annual_sales["Year"].eq(2025)].drop(columns="Year")
current = annual_sales.loc[annual_sales["Year"].eq(2026)].drop(columns="Year")
comparison = previous.merge(
    current, on=["Customer", "Product"], how="outer",
    suffixes=(" 2025", " 2026"), validate="one_to_one", indicator=True,
)
if not comparison["_merge"].eq("both").all():
    raise ValueError("New or lost relationships require separate volume/mix treatment.")
if not comparison[["Qty 2025", "Qty 2026"]].gt(0).all().all():
    raise ValueError("Positive quantities are required to calculate average prices.")

comparison["Avrg. price 2025"] = comparison["Sales 2025"] / comparison["Qty 2025"]
comparison["Avrg. price 2026"] = comparison["Sales 2026"] / comparison["Qty 2026"]
comparison["Price effect"] = (
    (comparison["Avrg. price 2026"] - comparison["Avrg. price 2025"])
    * comparison["Qty 2025"]
)
comparison["Volume mix effect"] = (
    comparison["Avrg. price 2026"]
    * (comparison["Qty 2026"] - comparison["Qty 2025"])
)

columns = [
    "Customer", "Product", "Sales 2025", "Sales 2026", "Qty 2025", "Qty 2026",
    "Avrg. price 2025", "Avrg. price 2026", "Price effect", "Volume mix effect",
]
price_volume_table = (
    comparison[columns]
    .sort_values(["Customer", "Product"])
    .reset_index(drop=True)
)

# Check every relationship before rounding the displayed values.
reconciliation_error = (
    price_volume_table["Sales 2026"] - price_volume_table["Sales 2025"]
    - price_volume_table["Price effect"] - price_volume_table["Volume mix effect"]
)
assert reconciliation_error.abs().lt(1e-8).all(), "Sales bridge does not reconcile."

money_columns = [
    "Sales 2025", "Sales 2026", "Avrg. price 2025", "Avrg. price 2026",
    "Price effect", "Volume mix effect",
]
price_volume_table.style.format(
    {**{column: "{:,.2f}" for column in money_columns},
     "Qty 2025": "{:,.0f}", "Qty 2026": "{:,.0f}"}
).hide(axis="index")

Customer,Product,Sales 2025,Sales 2026,Qty 2025,Qty 2026,Avrg. price 2025,Avrg. price 2026,Price effect,Volume mix effect
Customer A,Folder,240.00,252.00,80,90,3.00,2.80,-16.00,28.00
Customer A,Notebook,500.00,660.00,100,120,5.00,5.50,50.00,110.00
Customer A,Pen,400.00,396.00,200,180,2.00,2.20,40.00,-44.00
Customer B,Folder,300.00,280.00,100,100,3.00,2.80,-20.00,0.00
Customer B,Notebook,750.00,770.00,150,140,5.00,5.50,75.00,-55.00
Customer B,Pen,600.00,726.00,300,330,2.00,2.20,60.00,66.00
Customer C,Folder,180.00,140.00,60,50,3.00,2.80,-12.00,-28.00
Customer C,Notebook,400.00,550.00,80,100,5.00,5.50,40.00,110.00
Customer C,Pen,300.00,330.00,150,150,2.00,2.20,30.00,0.00
Customer D,Folder,270.00,294.00,90,105,3.00,2.80,-18.00,42.00


In [3]:
# Sum relationship-level effects to reconcile the overall revenue change.
total_sales_2025 = price_volume_table["Sales 2025"].sum()
total_sales_2026 = price_volume_table["Sales 2026"].sum()
total_price_effect = price_volume_table["Price effect"].sum()
total_volume_mix_effect = price_volume_table["Volume mix effect"].sum()
sales_change = total_sales_2026 - total_sales_2025
assert abs(sales_change - total_price_effect - total_volume_mix_effect) < 1e-8

print(f"Sales 2025:        {total_sales_2025:,.2f}")
print(f"Sales 2026:        {total_sales_2026:,.2f}")
print(f"Sales change:     {sales_change:,.2f}")
print(f"Price effect:     {total_price_effect:,.2f}")
print(f"Volume mix effect:{total_volume_mix_effect:,.2f}")
print("Reconciliation passed for every customer–product pair and overall.")

Sales 2025:        5,040.00
Sales 2026:        5,619.00
Sales change:     579.00
Price effect:     339.00
Volume mix effect:240.00
Reconciliation passed for every customer–product pair and overall.
